# Interaction Terms

Model effect modification with interactions.

## Example 1: Continuous × Binary Interaction

When one variable is continuous and the other is binary, the interaction allows different slopes for each group.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from aurora.models import fit_glm

# Set random seed for reproducibility
np.random.seed(42)

# Generate data with interaction effect
n = 300
x1_ex1 = np.random.randn(n)
x2_ex1 = np.random.binomial(1, 0.5, n)

# True model: y = 5 + 2*x1 + 3*x2 + 1.5*x1*x2 + noise
# This means:
# - When x2=0: y = 5 + 2*x1
# - When x2=1: y = 8 + 3.5*x1
# The slope for x1 changes depending on x2 (interaction effect)
y_ex1 = 5 + 2*x1_ex1 + 3*x2_ex1 + 1.5*x1_ex1*x2_ex1 + np.random.randn(n)

# Create design matrix WITHOUT intercept (fit_glm will add it)
X_with_interaction_ex1 = np.column_stack([x1_ex1, x2_ex1, x1_ex1*x2_ex1])
X_no_interaction_ex1 = np.column_stack([x1_ex1, x2_ex1])

# Fit model WITH interaction
result_interaction_ex1 = fit_glm(X_with_interaction_ex1, y_ex1, family='gaussian')

# Fit model WITHOUT interaction
result_no_interaction_ex1 = fit_glm(X_no_interaction_ex1, y_ex1, family='gaussian')

print('=' * 60)
print('MODEL WITH INTERACTION')
print('=' * 60)
print(f'Intercept: {result_interaction_ex1.intercept_:.3f}  (true: 5.0)')
print(f'X1:        {result_interaction_ex1.coef_[0]:.3f}  (true: 2.0)')
print(f'X2:        {result_interaction_ex1.coef_[1]:.3f}  (true: 3.0)')
print(f'X1:X2:     {result_interaction_ex1.coef_[2]:.3f}  (true: 1.5)')
print(f'AIC: {result_interaction_ex1.aic_:.2f}')
print(f'BIC: {result_interaction_ex1.bic_:.2f}')

print('\n' + '=' * 60)
print('MODEL WITHOUT INTERACTION')
print('=' * 60)
print(f'Intercept: {result_no_interaction_ex1.intercept_:.3f}')
print(f'X1:        {result_no_interaction_ex1.coef_[0]:.3f}')
print(f'X2:        {result_no_interaction_ex1.coef_[1]:.3f}')
print(f'AIC: {result_no_interaction_ex1.aic_:.2f}')
print(f'BIC: {result_no_interaction_ex1.bic_:.2f}')

print('\n' + '=' * 60)
print('INTERPRETATION')
print('=' * 60)
print(f'When X2=0: slope for X1 = {result_interaction_ex1.coef_[0]:.3f}')
print(f'When X2=1: slope for X1 = {result_interaction_ex1.coef_[0] + result_interaction_ex1.coef_[2]:.3f}')
print(f'Difference in slopes: {result_interaction_ex1.coef_[2]:.3f}')
print(f'\nLower AIC/BIC confirms interaction model is better')

## Key Concepts

### When to use interactions:
1. **Effect modification**: The effect of X1 depends on the level of X2
2. **Synergistic effects**: Combined effect differs from sum of individual effects
3. **Stratified relationships**: Different groups have different slopes

### Interpretation:
- **X1:X2 coefficient**: How much the effect of X1 changes per unit increase in X2
- **For categorical × continuous**: Different slopes for each category
- **For continuous × continuous**: Effect varies smoothly across the range

### Model comparison:
- Use **AIC/BIC** to compare models with and without interactions
- Check **residual plots** for patterns that suggest missing interactions
- Test **interaction coefficient** for statistical significance

In [ ]:
# Generate data with continuous × continuous interaction
np.random.seed(123)
n = 400
x1_ex2 = np.random.uniform(-2, 2, n)
x2_ex2 = np.random.uniform(-2, 2, n)

# True model: y = 10 + 2*x1 + 3*x2 + 0.8*x1*x2 + noise
y_ex2 = 10 + 2*x1_ex2 + 3*x2_ex2 + 0.8*x1_ex2*x2_ex2 + np.random.randn(n)

# Fit with interaction
X_ex2 = np.column_stack([x1_ex2, x2_ex2, x1_ex2*x2_ex2])
result_ex2 = fit_glm(X_ex2, y_ex2, family='gaussian')

print('Continuous × Continuous Interaction')
print('=' * 60)
print(f'Intercept: {result_ex2.intercept_:.3f}  (true: 10.0)')
print(f'X1:        {result_ex2.coef_[0]:.3f}  (true: 2.0)')
print(f'X2:        {result_ex2.coef_[1]:.3f}  (true: 3.0)')
print(f'X1:X2:     {result_ex2.coef_[2]:.3f}  (true: 0.8)')

# Create 3D surface plot
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(14, 5))

# Plot 1: 3D scatter and surface
ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(x1_ex2, x2_ex2, y_ex2, alpha=0.3, s=20, c='blue')

# Create prediction surface
x1_grid = np.linspace(-2, 2, 30)
x2_grid = np.linspace(-2, 2, 30)
X1_mesh, X2_mesh = np.meshgrid(x1_grid, x2_grid)
X_mesh = np.column_stack([X1_mesh.ravel(), X2_mesh.ravel(), 
                          (X1_mesh * X2_mesh).ravel()])
Y_mesh = result_ex2.predict(X_mesh).reshape(X1_mesh.shape)

ax1.plot_surface(X1_mesh, X2_mesh, Y_mesh, alpha=0.6, cmap='viridis')
ax1.set_xlabel('X1', fontsize=10)
ax1.set_ylabel('X2', fontsize=10)
ax1.set_zlabel('Y', fontsize=10)
ax1.set_title('3D Surface with Interaction', fontsize=12, fontweight='bold')

# Plot 2: Contour plot
ax2 = fig.add_subplot(122)
contour = ax2.contourf(X1_mesh, X2_mesh, Y_mesh, levels=15, cmap='viridis', alpha=0.8)
ax2.scatter(x1_ex2, x2_ex2, c=y_ex2, s=20, edgecolors='black', linewidth=0.5, cmap='viridis')
plt.colorbar(contour, ax=ax2, label='Y')
ax2.set_xlabel('X1', fontsize=12)
ax2.set_ylabel('X2', fontsize=12)
ax2.set_title('Contour Plot with Interaction', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\nThe curved/twisted surface shows the interaction effect')
print('Effect of X1 on Y changes as X2 changes (and vice versa)')

## Example 2: Continuous × Continuous Interaction

When both variables are continuous, the interaction allows the effect of one variable to change smoothly with the other.

In [ ]:
# Visualize the interaction effect
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Data with fitted lines
ax1 = axes[0]
# Separate data by x2
mask_0 = (x2_ex1 == 0)
mask_1 = (x2_ex1 == 1)

ax1.scatter(x1_ex1[mask_0], y_ex1[mask_0], alpha=0.5, s=50, label='X2 = 0', color='blue')
ax1.scatter(x1_ex1[mask_1], y_ex1[mask_1], alpha=0.5, s=50, label='X2 = 1', color='red')

# Create prediction lines
x1_range = np.linspace(x1_ex1.min(), x1_ex1.max(), 100)

# Predictions for x2=0 (with interaction model)
X_pred_0 = np.column_stack([x1_range, np.zeros(100), np.zeros(100)])
y_pred_0 = result_interaction_ex1.predict(X_pred_0)
ax1.plot(x1_range, y_pred_0, 'b-', linewidth=2.5, label='Fit: X2=0')

# Predictions for x2=1 (with interaction model)
X_pred_1 = np.column_stack([x1_range, np.ones(100), x1_range])
y_pred_1 = result_interaction_ex1.predict(X_pred_1)
ax1.plot(x1_range, y_pred_1, 'r-', linewidth=2.5, label='Fit: X2=1')

ax1.set_xlabel('X1', fontsize=12)
ax1.set_ylabel('Y', fontsize=12)
ax1.set_title('Interaction Effect: Different Slopes for X1', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Plot 2: Residuals comparison
ax2 = axes[1]
residuals_with = y_ex1 - result_interaction_ex1.predict(X_with_interaction_ex1)
residuals_without = y_ex1 - result_no_interaction_ex1.predict(X_no_interaction_ex1)

ax2.scatter(result_interaction_ex1.predict(X_with_interaction_ex1), residuals_with, 
           alpha=0.5, s=50, label='With interaction', color='green')
ax2.scatter(result_no_interaction_ex1.predict(X_no_interaction_ex1), residuals_without, 
           alpha=0.5, s=50, label='Without interaction', color='orange')
ax2.axhline(y=0, color='black', linestyle='--', linewidth=1)
ax2.set_xlabel('Fitted values', fontsize=12)
ax2.set_ylabel('Residuals', fontsize=12)
ax2.set_title('Residual Comparison', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate residual standard errors
rse_with = np.sqrt(np.mean(residuals_with**2))
rse_without = np.sqrt(np.mean(residuals_without**2))

print(f'\nResidual Standard Error:')
print(f'  With interaction:    {rse_with:.4f}')
print(f'  Without interaction: {rse_without:.4f}')
print(f'  Improvement: {((rse_without - rse_with) / rse_without * 100):.1f}%')